# 01_resource — Resource as pure transport, separate from the Action

AOA splits two beings ordinary services blur together: an **Action** has no memory (every call starts clean, everything arrives via `Params`/`state`/dependencies — it holds the *decisions*), while a **Resource** has long-lived state (a connection, pool, client, store living across calls — it holds the *transport* only). Golden rule: state outlives one call → `Resource`; state lives only inside the call → `Action`.

Because the decision lives in the Action and the transport in the Resource, the Action is tested by swapping the Resource — same operation, different store.

**What's new**

| Concept | Description |
|---------|-------------|
| `BaseResource` + `@meta` | a resource declares description + domain |
| `get_wrapper_class()` | `None` = direct pass-through; a transactional resource returns a wrapper |
| transport only | no business `if` in the resource; the decision is in the Action |
| swap to test | same Action, a different store behind `@connection` |

In [ ]:
!pip install aoa-action-machine

In [ ]:
import asyncio

from pydantic import Field

from aoa.action_machine.auth import NoneRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.connection import connection
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.resources.base_resource import BaseResource
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## The resource — transport only

`InventoryResource` can only read and write stock (`get_stock`, `take`). It returns `None` from `get_wrapper_class()` (in-memory → direct pass-through). No business rules live here.

In [ ]:
class WarehouseDomain(BaseDomain):
    name = "warehouse"
    description = "Warehouse domain"


@meta(description="In-memory inventory store (transport only)", domain=WarehouseDomain)
class InventoryResource(BaseResource):

    def __init__(self, stock: dict[str, int] | None = None) -> None:
        self._stock: dict[str, int] = dict(stock or {})

    def get_wrapper_class(self) -> type[BaseResource] | None:
        return None  # in-memory: direct pass-through; a SQL resource would return a wrapper

    async def get_stock(self, sku: str) -> int:
        return self._stock.get(sku, 0)

    async def take(self, sku: str, qty: int) -> None:
        self._stock[sku] = self._stock.get(sku, 0) - qty

## The action — the decision lives here

`ReserveStockAction` reads the resource by key and decides whether there is enough stock to reserve. The `if available < params.qty` lives in the Action, not the Resource.

In [ ]:
class ReserveParams(BaseParams):
    sku: str = Field(description="Product code")
    qty: int = Field(gt=0, description="Quantity to reserve")


class ReserveResult(BaseResult):
    reserved: bool = Field(description="Whether the reservation succeeded")
    remaining: int = Field(description="Stock left for this SKU")


@meta(description="Reserve stock for a SKU", domain=WarehouseDomain)
@check_roles(NoneRole)
@connection(InventoryResource, key="inventory")
class ReserveStockAction(BaseAction[ReserveParams, ReserveResult]):

    @summary_aspect("Reserve if enough stock")
    async def reserve_summary(self, params, state, box, connections):
        inventory = connections["inventory"]          # transport, supplied per request
        available = await inventory.get_stock(params.sku)
        # The DECISION lives here, in the Action — not in the Resource.
        if available < params.qty:
            return ReserveResult(reserved=False, remaining=available)
        await inventory.take(params.sku, params.qty)
        return ReserveResult(reserved=True, remaining=available - params.qty)

## Run

The decision "reserve or refuse" is the same; only the store behind the resource changes. Swapping in-memory → PostgreSQL means rewriting only the resource — the Action logic is untouched.

> In Colab, `await` works at top level — no `asyncio.run()`.

In [ ]:
async def main() -> None:
    machine = ActionProductMachine()

    print("Real inventory (sku-1: 10 in stock):")
    real = InventoryResource(stock={"sku-1": 10})
    for qty in (3, 99):
        r = await machine.run(Context(), ReserveStockAction(), ReserveParams(sku="sku-1", qty=qty),
                              connections={"inventory": real})
        print(f"  reserve {qty:<3} -> reserved={r.reserved}, remaining={r.remaining}")

    print("\nSwapped resource (empty store, same Action):")
    empty = InventoryResource(stock={})
    r = await machine.run(Context(), ReserveStockAction(), ReserveParams(sku="sku-1", qty=1),
                          connections={"inventory": empty})
    print(f"  reserve 1   -> reserved={r.reserved}, remaining={r.remaining}")


await main()